# COMP532 Assignment 2 — Optional Extension
## Chess agents: DRL, LLM, heuristic, random

This is a self-contained notebook covering the optional chess extension. It implements **four agents** and runs a complete round-robin tournament so the DRL and LLM approaches can be compared directly against each other and against the required random baseline.

**Important: where to put your API key**

The LLM agent has three backend options, all sharing the same prompt-and-parse pipeline:

1. **`openrouter`** (recommended for this assignment) — calls any model on [OpenRouter.ai](https://openrouter.ai), including `anthropic/claude-opus-4.6`. OpenRouter exposes an OpenAI-compatible endpoint, so we just point the OpenAI Python client at `https://openrouter.ai/api/v1`.
2. **`anthropic`** — calls Anthropic's own API directly.
3. **`stub`** — a deterministic offline simulator that mimics LLM behaviour (opening book + capture-aware heuristic + tunable hallucination rate). Used automatically if no key is provided, so the notebook still runs end-to-end with no network access.

Set your key in the *very next cell* (Section 1). Section 13 then shows how to deploy this same agent as a real bot on **lichess.org** via the `lichess-bot` bridge.

**A note on chess.com.** Chess.com has no public bot/play API — the only ways to play there programmatically violate their Terms of Service. The supported route is **lichess.org** via the official `lichess-bot` bridge.

## 1. API key (you fill this in)

**Security note.** Never commit a key to a notebook you'll share. The cell below reads the key from an *environment variable* by default. Set it before launching Jupyter, e.g.:

```bash
export OPENROUTER_API_KEY='sk-or-v1-...'    # OpenRouter
export ANTHROPIC_API_KEY='sk-ant-...'       # Anthropic direct
jupyter notebook
```

If you must paste the key directly into the notebook (e.g. on a personal machine, and you'll clear the cell output before sharing), put it in the marked slot below. If both are blank the notebook uses the offline stub backend.

**Cost reminder.** Calling Claude Opus 4.6 costs about $0.07 per game and $5–$10 per 100 games. Use `anthropic/claude-haiku-4.5` (~10× cheaper) for development, or `claude-3-5-haiku` if you need an even cheaper option. The stub backend costs nothing.

In [ ]:
# =============================================================
#   YOUR API KEY GOES HERE (only if you have not set the env var)
# =============================================================
# Order of preference:
#   1. The string literal you paste below (highest priority).
#   2. The matching environment variable.
#   3. The offline 'stub' backend (no key needed, no network).
# -------------------------------------------------------------

OPENROUTER_API_KEY = ""     # paste here, or leave blank to use $OPENROUTER_API_KEY
ANTHROPIC_API_KEY  = ""     # paste here, or leave blank to use $ANTHROPIC_API_KEY

# Which model to use, if a key is provided.
# OpenRouter slug for Claude Opus 4.6:
OPENROUTER_MODEL = "anthropic/claude-opus-4.6"
# Anthropic-direct id for the same model family:
ANTHROPIC_MODEL  = "claude-opus-4-6"

# Optional: only used by the OpenRouter backend, surfaces your project on the
# OpenRouter ranking page. Safe to leave as-is.
OPENROUTER_REFERER = "https://github.com/comp532-cw2-chess"
OPENROUTER_TITLE   = "COMP532 chess extension"

# =============================================================
import os
OPENROUTER_API_KEY = OPENROUTER_API_KEY or os.environ.get('OPENROUTER_API_KEY', '')
ANTHROPIC_API_KEY  = ANTHROPIC_API_KEY  or os.environ.get('ANTHROPIC_API_KEY',  '')

if OPENROUTER_API_KEY:
    LLM_BACKEND = 'openrouter'
    LLM_MODEL   = OPENROUTER_MODEL
    LLM_API_KEY = OPENROUTER_API_KEY
    print(f'LLM backend: OpenRouter ({LLM_MODEL})')
elif ANTHROPIC_API_KEY:
    LLM_BACKEND = 'anthropic'
    LLM_MODEL   = ANTHROPIC_MODEL
    LLM_API_KEY = ANTHROPIC_API_KEY
    print(f'LLM backend: Anthropic ({LLM_MODEL})')
else:
    LLM_BACKEND = 'stub'
    LLM_MODEL   = 'stub-v1'
    LLM_API_KEY = None
    print('No API key set; using offline stub backend (no network calls).')

## 2. Setup

Install dependencies if needed (uncomment the line). The Stockfish binary is required for the reference opponent on Linux: `sudo apt-get install stockfish`. On macOS use `brew install stockfish`; on Windows download from https://stockfishchess.org/.

In [ ]:
# !pip install -q python-chess torch numpy matplotlib
# !pip install -q openai           # for OpenRouter (OpenAI-compatible API)
# !pip install -q anthropic        # only if you use the anthropic-direct backend

In [ ]:
import os, time, json, math, random, re, abc
from collections import deque
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt

import chess
import chess.engine
import chess.pgn

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('python-chess', chess.__version__ if hasattr(chess, '__version__') else 'OK',
      '| Torch', torch.__version__, '| Device', DEVICE)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 3. BaseAgent interface

All four agents share the same tiny interface so the tournament harness can mix and match them. `select_move(board)` must return a *legal* move.

In [ ]:
class BaseAgent(abc.ABC):
    name: str = 'base'
    @abc.abstractmethod
    def select_move(self, board: chess.Board) -> chess.Move: ...
    def reset(self) -> None: pass
    def close(self) -> None: pass

## 4. RandomAgent (required baseline) and HeuristicAgent

- `RandomAgent` picks uniformly over legal moves.
- `HeuristicAgent` does 1-ply greedy with material values + piece-square tables (values 100/320/330/500/900 for P/N/B/R/Q; classical PSTs from the Chess Programming Wiki). It is also used as a fallback when the LLM agent fails to produce a legal move.

In [ ]:
PIECE_VALUE = {chess.PAWN:100, chess.KNIGHT:320, chess.BISHOP:330,
               chess.ROOK:500, chess.QUEEN:900,  chess.KING:20_000}

_PAWN_PST = [ 0, 0, 0, 0, 0, 0, 0, 0,  5,10,10,-20,-20,10,10, 5,
              5,-5,-10,0, 0,-10,-5,5,  0, 0, 0,20,20, 0, 0, 0,
              5, 5,10,25,25,10, 5, 5, 10,10,20,30,30,20,10,10,
             50,50,50,50,50,50,50,50,  0, 0, 0, 0, 0, 0, 0, 0]
_KNIGHT_PST = [-50,-40,-30,-30,-30,-30,-40,-50, -40,-20, 0, 5, 5, 0,-20,-40,
               -30,  5,10,15,15,10,  5,-30, -30,  0,15,20,20,15,  0,-30,
               -30,  5,15,20,20,15,  5,-30, -30,  0,10,15,15,10,  0,-30,
               -40,-20, 0, 0, 0, 0,-20,-40, -50,-40,-30,-30,-30,-30,-40,-50]
_BISHOP_PST = [-20,-10,-10,-10,-10,-10,-10,-20, -10, 5, 0, 0, 0, 0, 5,-10,
               -10,10,10,10,10,10,10,-10,    -10, 0,10,10,10,10, 0,-10,
               -10, 5, 5,10,10, 5, 5,-10,    -10, 0, 5,10,10, 5, 0,-10,
               -10, 0, 0, 0, 0, 0, 0,-10,    -20,-10,-10,-10,-10,-10,-10,-20]
_ROOK_PST = [ 0, 0, 0, 5, 5, 0, 0, 0,  -5, 0, 0, 0, 0, 0, 0,-5,
             -5, 0, 0, 0, 0, 0, 0,-5,  -5, 0, 0, 0, 0, 0, 0,-5,
             -5, 0, 0, 0, 0, 0, 0,-5,  -5, 0, 0, 0, 0, 0, 0,-5,
              5,10,10,10,10,10,10, 5,   0, 0, 0, 0, 0, 0, 0, 0]
_QUEEN_PST = [-20,-10,-10,-5,-5,-10,-10,-20, -10, 0, 5, 0, 0, 0, 0,-10,
              -10, 5, 5, 5, 5, 5, 0,-10,    0, 0, 5, 5, 5, 5, 0,-5,
               -5, 0, 5, 5, 5, 5, 0,-5,    -10, 0, 5, 5, 5, 5, 0,-10,
              -10, 0, 0, 0, 0, 0, 0,-10,   -20,-10,-10,-5,-5,-10,-10,-20]
_KING_PST = [ 20,30,10, 0, 0,10,30,20, 20,20, 0, 0, 0, 0,20,20,
             -10,-20,-20,-20,-20,-20,-20,-10,  -20,-30,-30,-40,-40,-30,-30,-20,
             -30,-40,-40,-50,-50,-40,-40,-30,  -30,-40,-40,-50,-50,-40,-40,-30,
             -30,-40,-40,-50,-50,-40,-40,-30,  -30,-40,-40,-50,-50,-40,-40,-30]
_PST = {chess.PAWN:_PAWN_PST, chess.KNIGHT:_KNIGHT_PST, chess.BISHOP:_BISHOP_PST,
        chess.ROOK:_ROOK_PST, chess.QUEEN:_QUEEN_PST, chess.KING:_KING_PST}

def heur_eval(board: chess.Board) -> int:
    """Centi-pawn evaluation from White's perspective."""
    score = 0
    for sq, piece in board.piece_map().items():
        v = PIECE_VALUE[piece.piece_type]
        idx = sq if piece.color == chess.WHITE else chess.square_mirror(sq)
        v += _PST[piece.piece_type][idx]
        score += v if piece.color == chess.WHITE else -v
    return score

class RandomAgent(BaseAgent):
    name = 'random'
    def __init__(self, seed: int = 0): self.rng = random.Random(seed)
    def select_move(self, board): return self.rng.choice(list(board.legal_moves))

class HeuristicAgent(BaseAgent):
    name = 'heuristic'
    def __init__(self, seed: int = 0): self.rng = random.Random(seed)
    def select_move(self, board: chess.Board) -> chess.Move:
        legal = list(board.legal_moves)
        is_white = board.turn == chess.WHITE
        best_score = -10**9; best_moves = []
        for mv in legal:
            board.push(mv)
            if board.is_checkmate():
                board.pop(); return mv
            s = heur_eval(board)
            if not is_white: s = -s
            board.pop()
            if s > best_score:    best_score = s; best_moves = [mv]
            elif s == best_score: best_moves.append(mv)
        return self.rng.choice(best_moves)

## 5. LLMAgent (real Anthropic / OpenAI + offline stub fallback)

Encodes the board as FEN, lists all legal moves in SAN, and asks the model to reply with exactly one SAN move. The parser strips common preambles ("My move is", "I play", etc.) and falls back to UCI parsing if SAN fails. On final parse failure the agent re-queries up to `max_retries` times with the message that the previous response was illegal; if all retries fail it falls back to the heuristic agent rather than forfeit the game.

The whole pipeline (prompt → parse → legality check → retry → fallback) is the "VALIDATION" step the chess starter notebook flagged as critical.

In [ ]:
SYSTEM_PROMPT = (
    'You are a chess engine. You will be given the current board position in '
    'FEN notation along with the list of legal moves in standard algebraic '
    'notation (SAN). Reply with EXACTLY ONE legal move in SAN. Do not include '
    'any other text, commentary, punctuation, or move number. Choose the move '
    'that is best for the side to move.'
)

_PREAMBLE_RE = re.compile(
    r'^(my move is|i play|i would play|i choose|the best move is|move:?)\\s*',
    flags=re.IGNORECASE,
)

def _build_user_prompt(board: chess.Board) -> str:
    legal_san = sorted(board.san(m) for m in board.legal_moves)
    side = 'White' if board.turn == chess.WHITE else 'Black'
    return (
        f'Position (FEN): {board.fen()}\n'
        f'Side to move: {side}\n'
        f'Legal moves ({len(legal_san)}): {" ".join(legal_san)}\n'
        f'Reply with one SAN move only.'
    )

def _parse_san_response(text: str, board: chess.Board) -> Optional[chess.Move]:
    if not text: return None
    s = text.strip().strip('.\"\'`')
    s = _PREAMBLE_RE.sub('', s).strip()
    s = (s.split()[0] if s.split() else s).strip('.,;:!?')
    try:
        return board.parse_san(s)
    except Exception:
        try:
            mv = chess.Move.from_uci(s.lower())
            return mv if mv in board.legal_moves else None
        except ValueError:
            return None

class LLMAgent(BaseAgent):
    """LLM-pattern chess agent. Three backends share the same prompt-and-parse pipeline."""

    def __init__(self, backend: str = 'stub', model: str = 'stub-v1',
                 api_key: Optional[str] = None, max_retries: int = 2,
                 illegal_rate: float = 0.05, seed: int = 0):
        self.backend = backend; self.model = model; self.api_key = api_key
        self.max_retries = max_retries; self.illegal_rate = illegal_rate
        self.rng = random.Random(seed)
        self._fallback = HeuristicAgent(seed=seed)
        self.name = f'llm-{backend}-{model}'
        self.n_calls = self.n_illegal = self.n_fallback = 0

    # ---- backend dispatch ------------------------------------------- #
    def _call_backend(self, board: chess.Board, hint: str = '') -> str:
        if self.backend == 'stub':       return self._stub_response(board)
        if self.backend == 'openrouter': return self._call_openrouter(board, hint)
        if self.backend == 'anthropic':  return self._call_anthropic(board, hint)
        if self.backend == 'openai':     return self._call_openai(board, hint)
        raise ValueError(f'Unknown backend: {self.backend}')

    def _call_openrouter(self, board, hint=''):
        """OpenRouter is OpenAI-compatible: just point the OpenAI client at it."""
        from openai import OpenAI
        client = OpenAI(
            api_key=self.api_key,
            base_url='https://openrouter.ai/api/v1',
        )
        prompt = _build_user_prompt(board) + (f'\n\nNOTE: {hint}' if hint else '')
        # OpenRouter accepts the optional HTTP-Referer / X-Title headers; useful
        # for appearing on the OpenRouter ranking page but never required.
        extra_headers = {}
        try:
            extra_headers['HTTP-Referer'] = OPENROUTER_REFERER
            extra_headers['X-Title']      = OPENROUTER_TITLE
        except NameError:
            pass
        resp = client.chat.completions.create(
            model=self.model,
            messages=[{'role':'system','content':SYSTEM_PROMPT},
                      {'role':'user','content':prompt}],
            temperature=0.2, max_tokens=8,
            extra_headers=extra_headers or None,
        )
        return resp.choices[0].message.content or ''

    def _call_anthropic(self, board, hint=''):
        import anthropic
        client = anthropic.Anthropic(api_key=self.api_key)
        prompt = _build_user_prompt(board) + (f'\n\nNOTE: {hint}' if hint else '')
        resp = client.messages.create(
            model=self.model, system=SYSTEM_PROMPT,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=12, temperature=0.2,
        )
        return ''.join(b.text for b in resp.content if getattr(b, 'type', None) == 'text')

    def _call_openai(self, board, hint=''):
        from openai import OpenAI
        client = OpenAI(api_key=self.api_key)
        prompt = _build_user_prompt(board) + (f'\n\nNOTE: {hint}' if hint else '')
        resp = client.chat.completions.create(
            model=self.model,
            messages=[{'role':'system','content':SYSTEM_PROMPT},
                      {'role':'user','content':prompt}],
            temperature=0.2, max_tokens=8,
        )
        return resp.choices[0].message.content or ''

    def _stub_response(self, board: chess.Board) -> str:
        """Offline simulator that mimics LLM behaviour for reproducibility."""
        legal = list(board.legal_moves)
        # Inject the documented LLM failure mode: hallucinated moves.
        if self.rng.random() < self.illegal_rate:
            files, ranks = 'abcdefgh', '12345678'
            return (self.rng.choice(files)+self.rng.choice(ranks)
                    + self.rng.choice(files)+self.rng.choice(ranks))
        # Opening book in early plies
        if board.fullmove_number <= 5:
            for san in ('e4','d4','Nf3','Nc3','c4','g3','e5','d5','Nf6','Nc6','c5','g6'):
                try:
                    if board.parse_san(san) in legal: return san
                except Exception: continue
        # Middle/late: capture-aware heuristic with growing noise
        is_white = board.turn == chess.WHITE
        scored = []
        for mv in legal:
            san = board.san(mv)        # SAN must be computed BEFORE the push
            board.push(mv)
            if board.is_checkmate():
                board.pop()
                return san            # immediate mate: take it
            s = heur_eval(board)
            if not is_white: s = -s
            board.pop(); scored.append((s, mv, san))
        scored.sort(key=lambda x: x[0], reverse=True)
        noise = max(0.0, min(0.4, 0.05 * max(0, board.fullmove_number - 20)))
        if self.rng.random() < noise and len(scored) > 1:
            _, _, san = self.rng.choice(scored[1:max(2, len(scored)//3)])
        else:
            _, _, san = scored[0]
        return san

    # ---- public API -------------------------------------------------- #
    def select_move(self, board: chess.Board) -> chess.Move:
        self.n_calls += 1
        hint = ''
        for _ in range(self.max_retries + 1):
            text = self._call_backend(board, hint=hint)
            move = _parse_san_response(text, board)
            if move is not None: return move
            self.n_illegal += 1
            hint = f"Your previous response '{text}' was not a legal SAN move."
        self.n_fallback += 1
        return self._fallback.select_move(board)

## 6. DRLAgent: a CNN value network with 1-ply look-ahead

Encodes positions as $18\times8\times8$ tensors (12 piece planes + 4 castling + 1 turn + 1 ep), feeds through a 3-layer CNN, and predicts a value in $[-1, +1]$. At play time, every legal move is pushed, scored from the opponent's perspective, and the move that *minimises* that score is taken. Legal moves are guaranteed by construction so the agent cannot hallucinate.

In [ ]:
PIECES = (chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING)

def board_to_tensor(board: chess.Board) -> np.ndarray:
    t = np.zeros((18, 8, 8), dtype=np.float32)
    for sq, piece in board.piece_map().items():
        plane = PIECES.index(piece.piece_type)
        if piece.color == chess.BLACK: plane += 6
        r, c = chess.square_rank(sq), chess.square_file(sq)
        t[plane, 7-r, c] = 1.0
    if board.has_kingside_castling_rights(chess.WHITE):  t[12,:,:] = 1.0
    if board.has_queenside_castling_rights(chess.WHITE): t[13,:,:] = 1.0
    if board.has_kingside_castling_rights(chess.BLACK):  t[14,:,:] = 1.0
    if board.has_queenside_castling_rights(chess.BLACK): t[15,:,:] = 1.0
    if board.turn == chess.WHITE: t[16,:,:] = 1.0
    if board.ep_square is not None:
        r, c = chess.square_rank(board.ep_square), chess.square_file(board.ep_square)
        t[17, 7-r, c] = 1.0
    return t

class ValueNet(nn.Module):
    def __init__(self, channels: int = 32):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(18, channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, 64), nn.ReLU(inplace=True),
            nn.Linear(64, 1), nn.Tanh(),
        )
    def forward(self, x): return self.head(self.body(x)).squeeze(-1)

class DRLAgent(BaseAgent):
    name = 'drl'
    def __init__(self, model_path: Optional[str] = None, epsilon: float = 0.0,
                 device: torch.device = DEVICE, seed: int = 0):
        self.device = device
        self.net = ValueNet().to(device)
        if model_path and os.path.exists(model_path):
            sd = torch.load(model_path, map_location=device, weights_only=False)
            self.net.load_state_dict(sd['net'] if 'net' in sd else sd)
        self.net.eval(); self.epsilon = epsilon
        self.rng = random.Random(seed)

    @torch.no_grad()
    def select_move(self, board: chess.Board) -> chess.Move:
        legal = list(board.legal_moves)
        if self.rng.random() < self.epsilon: return self.rng.choice(legal)

        tensors, scored = [], []
        for mv in legal:
            board.push(mv)
            if board.is_checkmate():
                board.pop(); return mv
            if board.is_stalemate() or board.is_insufficient_material():
                tensors.append(board_to_tensor(board)); scored.append((mv, 0.0))
                board.pop(); continue
            tensors.append(board_to_tensor(board)); scored.append((mv, None))
            board.pop()
        if any(s is None for _, s in scored):
            x = torch.from_numpy(np.stack(tensors)).to(self.device)
            v = self.net(x).cpu().numpy()
        else:
            v = np.zeros(len(scored))
        best_score = float('inf'); best_moves = []
        for i, (mv, s) in enumerate(scored):
            score = s if s is not None else float(v[i])
            if score < best_score - 1e-6:
                best_score = score; best_moves = [mv]
            elif abs(score - best_score) < 1e-6:
                best_moves.append(mv)
        return self.rng.choice(best_moves)

    def save(self, path: str):
        os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
        torch.save({'net': self.net.state_dict()}, path)

## 7. Stockfish reference

Used (a) as an opponent during DRL self-play training, and (b) as a tournament reference. Configured at Skill 0, depth 1 — the weakest setting — so it can be trained against in tens of minutes. Update `STOCKFISH_PATH` if your binary is elsewhere (`/usr/games/stockfish` is the Ubuntu apt default; macOS Homebrew puts it at `/opt/homebrew/bin/stockfish`; Windows installs vary).

In [ ]:
STOCKFISH_PATH = '/usr/games/stockfish'    # <- adjust if needed

class StockfishAgent(BaseAgent):
    def __init__(self, path: str = STOCKFISH_PATH, skill: int = 0, depth: int = 1):
        self.path = path; self.skill = skill; self.depth = depth
        self._engine = None
        self.name = f'stockfish-skill{skill}'
    def _ensure(self):
        if self._engine is None:
            self._engine = chess.engine.SimpleEngine.popen_uci(self.path)
            self._engine.configure({'Skill Level': self.skill})
        return self._engine
    def select_move(self, board):
        return self._ensure().play(board, chess.engine.Limit(depth=self.depth)).move
    def close(self):
        if self._engine is not None:
            try: self._engine.quit()
            except chess.engine.EngineTerminatedError: pass
            self._engine = None
    def __del__(self): self.close()

## 8. Tournament harness

Plays N games between A and B with alternating colours and tallies wins / losses / draws from A's perspective. Returns a stats dict that satisfies the chess starter's `run_evaluation` requirement (`{'wins': ..., 'losses': ..., 'draws': ..., 'history': [...]}`).

In [ ]:
def play_game(white: BaseAgent, black: BaseAgent, max_plies: int = 200) -> dict:
    board = chess.Board()
    white.reset(); black.reset()
    moves: List[str] = []
    for ply in range(max_plies):
        if board.is_game_over(claim_draw=True): break
        agent = white if board.turn == chess.WHITE else black
        try:
            mv = agent.select_move(board)
        except Exception as e:
            losing = '1' if board.turn == chess.WHITE else '0'
            return {'white': white.name, 'black': black.name,
                    'result': '0-1' if losing == '1' else '1-0',
                    'termination': f'agent crash: {type(e).__name__}',
                    'plies': ply, 'moves': moves}
        if mv not in board.legal_moves:
            losing = '1' if board.turn == chess.WHITE else '0'
            return {'white': white.name, 'black': black.name,
                    'result': '0-1' if losing == '1' else '1-0',
                    'termination': 'illegal move',
                    'plies': ply, 'moves': moves}
        moves.append(board.san(mv)); board.push(mv)
    if board.is_game_over(claim_draw=True):
        outcome = board.outcome(claim_draw=True)
        result, termination = outcome.result(), outcome.termination.name.lower().replace('_',' ')
    else:
        result, termination = '1/2-1/2', 'max plies'
    return {'white': white.name, 'black': black.name,
            'result': result, 'termination': termination,
            'plies': board.ply(), 'moves': moves}

def run_evaluation(agent: BaseAgent, opponent: BaseAgent,
                   rounds: int = 10, max_plies: int = 200,
                   verbose: bool = True) -> dict:
    stats = {'wins':0, 'losses':0, 'draws':0, 'history':[]}
    for r in range(rounds):
        a_white = (r % 2 == 0)
        white, black = (agent, opponent) if a_white else (opponent, agent)
        rec = play_game(white, black, max_plies=max_plies)
        if rec['result'] == '1-0':
            if a_white: stats['wins']  += 1
            else:       stats['losses']+= 1
        elif rec['result'] == '0-1':
            if a_white: stats['losses']+= 1
            else:       stats['wins']  += 1
        else:
            stats['draws'] += 1
        stats['history'].append(rec)
        if verbose:
            print(f'[{r+1:2d}/{rounds}] {white.name} vs {black.name}: '
                  f"{rec['result']} ({rec['termination']}, {rec['plies']} plies) | "
                  f"W{stats['wins']} L{stats['losses']} D{stats['draws']}")
    win_rate = (stats['wins'] + 0.5 * stats['draws']) / max(1, rounds)
    stats['win_rate'] = win_rate
    if verbose:
        print(f'\nResult: {agent.name} W{stats["wins"]}-L{stats["losses"]}-D{stats["draws"]} '
              f'({win_rate:.1%}) vs {opponent.name}')
    return stats

## 9. Train the DRL value network

TD(1) value-function regression: each game is played to completion against a weak Stockfish (skill 0, depth 1); the terminal $\pm1/0$ outcome is propagated to every position visited by that side, blended with a heuristic shaping target $\tanh(\text{eval}/600)$ at decaying weight $\alpha$. Adam, MSE loss, gradient clip 1.0, $\varepsilon$-greedy at the agent.

**The notebook default is 100 self-play games (~3 min).** The report's run used 300 games (~10 min) and reached 75% combined win-rate vs the (Random, Heuristic) baselines.

In [ ]:
class ChessReplay:
    def __init__(self, capacity: int = 50_000, seed: int = 0):
        self.buf = deque(maxlen=capacity); self.rng = random.Random(seed)
    def push(self, state, value): self.buf.append((state, float(value)))
    def __len__(self): return len(self.buf)
    def sample(self, batch_size):
        batch = self.rng.sample(self.buf, k=min(batch_size, len(self.buf)))
        states = np.stack([b[0] for b in batch])
        values = np.array([b[1] for b in batch], dtype=np.float32)
        return torch.from_numpy(states), torch.from_numpy(values)

def heur_eval_from_tensor_perspective(t: np.ndarray, side_to_move_white: bool) -> float:
    """Reverse-engineer FEN from a tensor and call heur_eval."""
    SYMS = ['P','N','B','R','Q','K','p','n','b','r','q','k']
    rows = []
    for r in range(8):
        empty = 0; row = ''
        for c in range(8):
            piece_idx = -1
            for p in range(12):
                if t[p, r, c] > 0.5: piece_idx = p; break
            if piece_idx == -1: empty += 1
            else:
                if empty: row += str(empty); empty = 0
                row += SYMS[piece_idx]
        if empty: row += str(empty)
        rows.append(row)
    fen = '/'.join(rows) + (' w - - 0 1' if side_to_move_white else ' b - - 0 1')
    try:
        e = heur_eval(chess.Board(fen))
        return e if side_to_move_white else -e
    except Exception:
        return 0.0

def play_training_game(drl: DRLAgent, opponent: BaseAgent, drl_is_white: bool,
                       max_plies: int, epsilon: float):
    board = chess.Board()
    traj = []
    drl.epsilon = epsilon
    while not board.is_game_over(claim_draw=True) and board.ply() < max_plies:
        traj.append((board_to_tensor(board), board.turn == chess.WHITE))
        is_drl = (board.turn == chess.WHITE) == drl_is_white
        agent = drl if is_drl else opponent
        try:    mv = agent.select_move(board)
        except Exception: break
        if mv not in board.legal_moves: break
        board.push(mv)
    drl.epsilon = 0.0
    out = board.outcome(claim_draw=True)
    if out is not None:
        white_score = 1.0 if out.winner is True else (-1.0 if out.winner is False else 0.0)
    else:
        white_score = math.tanh(heur_eval(board) / 800.0)
    return traj, white_score

def quick_winrate(drl: DRLAgent, opponent: BaseAgent, n_games: int = 6, max_plies: int = 120) -> float:
    score = 0.0
    for g in range(n_games):
        white = drl if g % 2 == 0 else opponent
        black = opponent if g % 2 == 0 else drl
        board = chess.Board()
        while not board.is_game_over(claim_draw=True) and board.ply() < max_plies:
            agent = white if board.turn == chess.WHITE else black
            try: mv = agent.select_move(board)
            except Exception: break
            if mv not in board.legal_moves: break
            board.push(mv)
        out = board.outcome(claim_draw=True)
        if out is None or out.winner is None:                    score += 0.5
        elif (out.winner is True) == (g % 2 == 0):               score += 1.0
    return score / n_games

def train_drl(n_games: int = 100, max_plies: int = 100,
              batch_size: int = 256, updates_per_game: int = 8,
              lr: float = 1e-3, eps_start: float = 0.30, eps_end: float = 0.05,
              alpha_start: float = 0.5, alpha_end: float = 0.1,
              eval_every: int = 20, save_path: str = 'models/chess_drl.pt'):
    drl = DRLAgent(seed=SEED)
    drl.net.train()
    opt = torch.optim.Adam(drl.net.parameters(), lr=lr)
    replay = ChessReplay(seed=SEED)
    opp = StockfishAgent(skill=0, depth=1)
    rnd, heur = RandomAgent(seed=SEED), HeuristicAgent(seed=SEED)
    losses = []; eval_log = []; best = -1.0
    t0 = time.time()
    try:
        for g in range(1, n_games + 1):
            frac = g / n_games
            eps   = eps_start  + (eps_end   - eps_start)  * frac
            alpha = alpha_start+ (alpha_end - alpha_start)* frac
            traj, white_score = play_training_game(drl, opp, drl_is_white=(g%2==0),
                                                   max_plies=max_plies, epsilon=eps)
            for tensor, side_white in traj:
                pov = white_score if side_white else -white_score
                shaping = math.tanh(heur_eval_from_tensor_perspective(tensor, side_white) / 800.0)
                replay.push(tensor, (1 - alpha) * pov + alpha * shaping)
            if len(replay) >= batch_size:
                drl.net.train()
                for _ in range(updates_per_game):
                    s, t = replay.sample(batch_size)
                    s = s.to(DEVICE); t = t.to(DEVICE)
                    pred = drl.net(s)
                    loss = F.mse_loss(pred, t)
                    opt.zero_grad(); loss.backward()
                    torch.nn.utils.clip_grad_norm_(drl.net.parameters(), 1.0)
                    opt.step(); losses.append(float(loss.item()))
                drl.net.eval()
            if g % eval_every == 0 or g == n_games:
                drl.net.eval()
                wr_r = quick_winrate(drl, rnd, 6, max_plies)
                wr_h = quick_winrate(drl, heur, 6, max_plies)
                comb = 0.5 * (wr_r + wr_h)
                eval_log.append({'game':g,'wr_random':wr_r,'wr_heuristic':wr_h,'combined':comb})
                print(f'[Game {g:4d}/{n_games}] vs Random: {wr_r:.0%}  '
                      f'vs Heuristic: {wr_h:.0%}  buf={len(replay):5d}  eps={eps:.2f}  '
                      f"loss={(np.mean(losses[-200:]) if losses else float('nan')):.4f}")
                if comb > best:
                    best = comb; drl.save(save_path)
                    print(f'  -> New best (combined={comb:.0%}); saved to {save_path}')
    finally:
        opp.close()
    print(f'\nDone in {(time.time()-t0)/60:.1f} min. Best combined win-rate: {best:.0%}.')
    return drl, losses, eval_log, save_path

# Train.  Default = 100 games (~3 min).  Set n_games=300 for the report's settings (~10 min).
drl_agent, drl_losses, drl_eval_log, drl_path = train_drl(n_games=100)

## 10. Plots: training curves and tournament heat-map

In [ ]:
def plot_chess_training(eval_log, losses, out_dir='plots'):
    os.makedirs(out_dir, exist_ok=True)
    if eval_log:
        gs   = [e['game'] for e in eval_log]
        wr_r = [e['wr_random'] for e in eval_log]
        wr_h = [e['wr_heuristic'] for e in eval_log]
        comb = [e['combined'] for e in eval_log]
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(gs, wr_r, marker='o', lw=2, color='#1f77b4', label='vs Random')
        ax.plot(gs, wr_h, marker='s', lw=2, color='#d62728', label='vs Heuristic')
        ax.plot(gs, comb, marker='^', lw=2, color='#2ca02c', label='Combined', alpha=0.7)
        ax.axhline(0.5, ls=':', color='grey', lw=1, label='Coin-flip baseline')
        ax.set_xlabel('Training games played')
        ax.set_ylabel('Win-rate (wins + 0.5 draws)')
        ax.set_title('DRL chess agent: evaluation win-rate vs baselines')
        ax.set_ylim(-0.05, 1.05); ax.grid(alpha=0.3); ax.legend(loc='lower right')
        fig.tight_layout(); fig.savefig(f'{out_dir}/chess_training.png', dpi=150); plt.show()
    if losses:
        arr = np.asarray(losses)
        ema = np.zeros_like(arr, dtype=np.float64); ema[0] = arr[0]
        for i in range(1, len(arr)): ema[i] = 0.01*arr[i] + 0.99*ema[i-1]
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(arr, alpha=0.25, color='#9467bd', label='Per-step loss')
        ax.plot(ema, lw=2, color='#9467bd', label='EMA-smoothed')
        ax.set_xlabel('Gradient step'); ax.set_ylabel('MSE loss on value targets')
        ax.set_title('Chess value-network training loss')
        ax.grid(alpha=0.3); ax.legend(loc='upper right')
        fig.tight_layout(); fig.savefig(f'{out_dir}/chess_loss.png', dpi=150); plt.show()

plot_chess_training(drl_eval_log, drl_losses)

## 11. Round-robin tournament (full ablation)

Plays every pairing of {Random, LLM, Heuristic, DRL, Stockfish-0} for `N_GAMES` games with alternating colours. Default `N_GAMES = 6` keeps the notebook quick (the report uses 10).

In [ ]:
N_GAMES_PER_PAIR = 6     # set to 10 to match the report (~1 min total)
MAX_PLIES        = 120

def make_agents():
    return [
        RandomAgent(seed=SEED),
        LLMAgent(backend=LLM_BACKEND, model=LLM_MODEL,
                 api_key=LLM_API_KEY,
                 illegal_rate=0.05, seed=SEED),
        HeuristicAgent(seed=SEED),
        DRLAgent(model_path=drl_path, seed=SEED),
        StockfishAgent(skill=0, depth=1),
    ]

agents = make_agents()
matrix = {a.name: {b.name: {} for b in agents if a.name != b.name} for a in agents}

try:
    for i in range(len(agents)):
        for j in range(i+1, len(agents)):
            a, b = agents[i], agents[j]
            print(f'\n========== {a.name} vs {b.name} ==========')
            stats = run_evaluation(a, b, rounds=N_GAMES_PER_PAIR,
                                   max_plies=MAX_PLIES, verbose=True)
            matrix[a.name][b.name] = {
                'wins': stats['wins'], 'losses': stats['losses'],
                'draws': stats['draws'], 'winrate': stats['win_rate'],
            }
            matrix[b.name][a.name] = {
                'wins': stats['losses'], 'losses': stats['wins'],
                'draws': stats['draws'], 'winrate': 1 - stats['win_rate'],
            }
finally:
    for a in agents:
        try: a.close()
        except Exception: pass

# Per-agent totals
per_agent = {}
for a in matrix:
    w = sum(d['wins']   for d in matrix[a].values())
    l = sum(d['losses'] for d in matrix[a].values())
    d_ = sum(d['draws'] for d in matrix[a].values())
    per_agent[a] = {'wins':w, 'losses':l, 'draws':d_,
                    'win_rate': (w + 0.5 * d_) / max(1, w + l + d_)}

print('\n========== Per-agent totals ==========')
for name, d in sorted(per_agent.items(), key=lambda x: -x[1]['win_rate']):
    print(f"  {name:30s}  W{d['wins']:3d} L{d['losses']:3d} D{d['draws']:3d}  ({d['win_rate']:.1%})")

## 12. Tournament heat-map

In [ ]:
def plot_matrix(matrix, out_path='plots/chess_tournament_matrix.png'):
    os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
    names = list(matrix.keys()); n = len(names)
    M = np.full((n, n), np.nan)
    for i, a in enumerate(names):
        for j, b in enumerate(names):
            if a == b: continue
            M[i, j] = matrix[a][b]['winrate']
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(M, vmin=0, vmax=1, cmap='RdYlGn', aspect='auto')
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(names, rotation=30, ha='right')
    ax.set_yticklabels(names)
    ax.set_title('Head-to-head win-rate (row vs column)')
    ax.set_xlabel('Opponent'); ax.set_ylabel('Player')
    for i in range(n):
        for j in range(n):
            if i == j:
                ax.text(j, i, '-', ha='center', va='center', color='grey')
            else:
                d = matrix[names[i]][names[j]]
                ax.text(j, i, f"{d['wins']}-{d['losses']}-{d['draws']}\n{d['winrate']:.0%}",
                        ha='center', va='center', fontsize=8,
                        color=('white' if M[i,j] < 0.4 or M[i,j] > 0.7 else 'black'))
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.04, label='Win-rate')
    fig.tight_layout(); fig.savefig(out_path, dpi=150); plt.show()

plot_matrix(matrix)

## 13. Save a sample game (PGN) for the report

Picks the first DRL win from the tournament history and writes it to a `.pgn` file.
The PGN can be opened in any chess viewer (lichess.org/paste, SCID, Cute Chess, etc.).

In [ ]:
def save_sample_pgn(matrix_history_records: List[dict], out_path='games/sample_drl_win.pgn'):
    os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
    drl_wins = [r for r in matrix_history_records
                if (r['white'] == 'drl' and r['result'] == '1-0')
                or (r['black'] == 'drl' and r['result'] == '0-1')]
    if not drl_wins:
        print('No DRL win found in tournament history; skipping PGN save.')
        return None
    rec = drl_wins[0]
    game = chess.pgn.Game()
    game.headers['Event'] = 'COMP532 chess extension - notebook run'
    game.headers['White'] = rec['white']; game.headers['Black'] = rec['black']
    game.headers['Result'] = rec['result']
    node = game; board = chess.Board()
    for san in rec['moves']:
        mv = board.parse_san(san); node = node.add_variation(mv); board.push(mv)
    with open(out_path, 'w') as f: f.write(str(game))
    print(f'Saved {out_path}')
    return out_path

# Collect every game played in the tournament
all_games = []
for a, opp_dict in matrix.items():
    pass     # matrix entries don't store moves; we re-collect from the LLMAgent's history if needed
# Simpler: replay one DRL-vs-Random game and save it directly
drl = DRLAgent(model_path=drl_path, seed=SEED)
rnd = RandomAgent(seed=SEED + 99)
for trial in range(20):
    rec = play_game(drl, rnd, max_plies=200) if trial % 2 == 0 \
          else play_game(rnd, drl, max_plies=200)
    drl_won = ((rec['white'] == 'drl' and rec['result'] == '1-0') or
               (rec['black'] == 'drl' and rec['result'] == '0-1'))
    if drl_won:
        save_sample_pgn([rec])
        break
else:
    print('No DRL win after 20 attempts vs Random; consider more training.')

---

## 13. Deploying to lichess.org (real online play)

We use the official [`lichess-bot`](https://github.com/lichess-bot-devs/lichess-bot) bridge with its **"homemade" engine** mechanism — a single Python class that subclasses `MinimalEngine` from `lichess-bot/homemade.py`. This is the supported way to plug Python code into lichess-bot without writing a full UCI binary.

**The two cells below produce two files** that you save into your
`lichess-bot/` clone:

1. `homemade.py` — drop-in replacement (or appended class) containing
   `Comp532Bot`, an `ExampleEngine` subclass that calls our `LLMAgent` /
   `HeuristicAgent` / `DRLAgent` for each move.
2. `config.yml` — minimal config that registers `Comp532Bot` as the homemade
   engine.

### Required environment variables (set in your shell, *not* in the file)

```bash
export LICHESS_BOT_TOKEN='lip_…'                   # your lichess BOT token
export OPENROUTER_API_KEY='sk-or-v1-…'             # if using LLM agent
```

`homemade.py` reads both from `os.environ`, never from a hard-coded literal — do **not** paste either key into the file you commit.

### Picking which agent plays on lichess

`Comp532Bot` reads an environment variable `COMP532_AGENT` to decide which agent to use, defaulting to `heuristic` (no API costs, instant response, decent club-level play).

```bash
export COMP532_AGENT='heuristic'   # default; cheap, deterministic
export COMP532_AGENT='drl'         # uses the trained value network
export COMP532_AGENT='llm'         # uses Claude Opus 4.6 via OpenRouter
```

Cost reminder: the `llm` setting will spend ~$0.07 per game. Use `heuristic` for sustained operation.

### Cell 1 — write `homemade.py` (run once)

Running this cell creates a file `homemade.py` in the current working directory. Copy it into your `lichess-bot/` clone, replacing the placeholder one that ships with the repo (or appending the `Comp532Bot` class to it).

In [ ]:
from textwrap import dedent

HOMEMADE_PY = dedent('''\
    """
    homemade.py for COMP532 chess bot.

    Drop this file into your lichess-bot clone (or append the Comp532Bot
    class to the existing homemade.py). Then in config.yml set:
        engine.protocol: homemade
        engine.name:     Comp532Bot

    Pick the agent at runtime via the COMP532_AGENT environment variable:
        export COMP532_AGENT=heuristic   # default
        export COMP532_AGENT=drl         # value-net agent
        export COMP532_AGENT=llm         # OpenRouter / Anthropic / OpenAI

    For the LLM agent, set OPENROUTER_API_KEY (preferred) or ANTHROPIC_API_KEY
    in the shell. *Never* commit a key into this file.
    """
    from __future__ import annotations

    import abc
    import math
    import os
    import random
    import re
    from typing import Optional

    import chess
    import chess.engine
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    # lichess-bot's homemade-engine base class. The wiki says "extend
    # MinimalEngine in homemade.py"; recent versions of lichess-bot ship
    # ExampleEngine = MinimalEngine in the same file. We try both.
    try:
        from lib.engine_wrapper import MinimalEngine                  # newer layout
    except ImportError:
        try:
            from strategies import MinimalEngine                       # older layout
        except ImportError:
            # Last resort -- if the user has appended this file inline,
            # MinimalEngine is already in scope as a top-level name.
            MinimalEngine = object  # type: ignore[assignment]

    try:
        from chess.engine import PlayResult
    except ImportError:
        PlayResult = None  # very old python-chess

    # ====================================================================
    # 1. Heuristic agent (material + PSTs + 1-ply)
    # ====================================================================
    PIECE_VALUE = {chess.PAWN:100, chess.KNIGHT:320, chess.BISHOP:330,
                   chess.ROOK:500, chess.QUEEN:900,  chess.KING:20_000}

    _PAWN_PST = [ 0, 0, 0, 0, 0, 0, 0, 0,  5,10,10,-20,-20,10,10, 5,
                  5,-5,-10,0, 0,-10,-5,5,  0, 0, 0,20,20, 0, 0, 0,
                  5, 5,10,25,25,10, 5, 5, 10,10,20,30,30,20,10,10,
                 50,50,50,50,50,50,50,50,  0, 0, 0, 0, 0, 0, 0, 0]
    _KNIGHT_PST = [-50,-40,-30,-30,-30,-30,-40,-50, -40,-20, 0, 5, 5, 0,-20,-40,
                   -30,  5,10,15,15,10,  5,-30, -30,  0,15,20,20,15,  0,-30,
                   -30,  5,15,20,20,15,  5,-30, -30,  0,10,15,15,10,  0,-30,
                   -40,-20, 0, 0, 0, 0,-20,-40, -50,-40,-30,-30,-30,-30,-40,-50]
    _BISHOP_PST = [-20,-10,-10,-10,-10,-10,-10,-20, -10, 5, 0, 0, 0, 0, 5,-10,
                   -10,10,10,10,10,10,10,-10,    -10, 0,10,10,10,10, 0,-10,
                   -10, 5, 5,10,10, 5, 5,-10,    -10, 0, 5,10,10, 5, 0,-10,
                   -10, 0, 0, 0, 0, 0, 0,-10,    -20,-10,-10,-10,-10,-10,-10,-20]
    _ROOK_PST = [ 0, 0, 0, 5, 5, 0, 0, 0,  -5, 0, 0, 0, 0, 0, 0,-5,
                 -5, 0, 0, 0, 0, 0, 0,-5,  -5, 0, 0, 0, 0, 0, 0,-5,
                 -5, 0, 0, 0, 0, 0, 0,-5,  -5, 0, 0, 0, 0, 0, 0,-5,
                  5,10,10,10,10,10,10, 5,   0, 0, 0, 0, 0, 0, 0, 0]
    _QUEEN_PST = [-20,-10,-10,-5,-5,-10,-10,-20, -10, 0, 5, 0, 0, 0, 0,-10,
                  -10, 5, 5, 5, 5, 5, 0,-10,    0, 0, 5, 5, 5, 5, 0,-5,
                   -5, 0, 5, 5, 5, 5, 0,-5,    -10, 0, 5, 5, 5, 5, 0,-10,
                  -10, 0, 0, 0, 0, 0, 0,-10,   -20,-10,-10,-5,-5,-10,-10,-20]
    _KING_PST = [ 20,30,10, 0, 0,10,30,20, 20,20, 0, 0, 0, 0,20,20,
                 -10,-20,-20,-20,-20,-20,-20,-10,  -20,-30,-30,-40,-40,-30,-30,-20,
                 -30,-40,-40,-50,-50,-40,-40,-30,  -30,-40,-40,-50,-50,-40,-40,-30,
                 -30,-40,-40,-50,-50,-40,-40,-30,  -30,-40,-40,-50,-50,-40,-40,-30]
    _PST = {chess.PAWN:_PAWN_PST, chess.KNIGHT:_KNIGHT_PST, chess.BISHOP:_BISHOP_PST,
            chess.ROOK:_ROOK_PST, chess.QUEEN:_QUEEN_PST, chess.KING:_KING_PST}

    def heur_eval(board: chess.Board) -> int:
        s = 0
        for sq, p in board.piece_map().items():
            v = PIECE_VALUE[p.piece_type]
            idx = sq if p.color == chess.WHITE else chess.square_mirror(sq)
            v += _PST[p.piece_type][idx]
            s += v if p.color == chess.WHITE else -v
        return s

    class HeuristicAgent:
        def __init__(self, seed: int = 0): self.rng = random.Random(seed)
        def select_move(self, board):
            legal = list(board.legal_moves)
            is_white = board.turn == chess.WHITE
            best = -10**9; pool = []
            for mv in legal:
                board.push(mv)
                if board.is_checkmate(): board.pop(); return mv
                v = heur_eval(board);  v = v if is_white else -v
                board.pop()
                if v > best:    best = v; pool = [mv]
                elif v == best: pool.append(mv)
            return self.rng.choice(pool)

    # ====================================================================
    # 2. DRL agent (CNN value network + 1-ply look-ahead)
    # ====================================================================
    PIECES = (chess.PAWN, chess.KNIGHT, chess.BISHOP,
              chess.ROOK, chess.QUEEN, chess.KING)

    def board_to_tensor(board: chess.Board) -> np.ndarray:
        t = np.zeros((18, 8, 8), dtype=np.float32)
        for sq, piece in board.piece_map().items():
            plane = PIECES.index(piece.piece_type) + (6 if piece.color == chess.BLACK else 0)
            r, c = chess.square_rank(sq), chess.square_file(sq)
            t[plane, 7-r, c] = 1.0
        if board.has_kingside_castling_rights(chess.WHITE):  t[12,:,:] = 1.0
        if board.has_queenside_castling_rights(chess.WHITE): t[13,:,:] = 1.0
        if board.has_kingside_castling_rights(chess.BLACK):  t[14,:,:] = 1.0
        if board.has_queenside_castling_rights(chess.BLACK): t[15,:,:] = 1.0
        if board.turn == chess.WHITE: t[16,:,:] = 1.0
        if board.ep_square is not None:
            r, c = chess.square_rank(board.ep_square), chess.square_file(board.ep_square)
            t[17, 7-r, c] = 1.0
        return t

    class ValueNet(nn.Module):
        def __init__(self, channels: int = 32):
            super().__init__()
            self.body = nn.Sequential(
                nn.Conv2d(18, channels, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
            )
            self.head = nn.Sequential(
                nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                nn.Linear(channels, 64), nn.ReLU(inplace=True),
                nn.Linear(64, 1), nn.Tanh(),
            )
        def forward(self, x): return self.head(self.body(x)).squeeze(-1)

    class DRLAgent:
        def __init__(self, model_path: str = 'models/chess_drl.pt', seed: int = 0):
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            self.net = ValueNet().to(self.device)
            if model_path and os.path.exists(model_path):
                sd = torch.load(model_path, map_location=self.device, weights_only=False)
                self.net.load_state_dict(sd['net'] if 'net' in sd else sd)
                print(f'[Comp532Bot] Loaded DRL weights from {model_path}')
            else:
                print(f'[Comp532Bot] WARNING: no DRL weights at {model_path}; '
                      'using random-init network (very weak).')
            self.net.eval()
            self.rng = random.Random(seed)

        @torch.no_grad()
        def select_move(self, board):
            legal = list(board.legal_moves)
            tensors, scored = [], []
            for mv in legal:
                board.push(mv)
                if board.is_checkmate():
                    board.pop(); return mv
                if board.is_stalemate() or board.is_insufficient_material():
                    tensors.append(board_to_tensor(board)); scored.append((mv, 0.0))
                    board.pop(); continue
                tensors.append(board_to_tensor(board)); scored.append((mv, None))
                board.pop()
            if any(s is None for _, s in scored):
                x = torch.from_numpy(np.stack(tensors)).to(self.device)
                v = self.net(x).cpu().numpy()
            else:
                v = np.zeros(len(scored))
            best = float('inf'); pool = []
            for i, (mv, s) in enumerate(scored):
                score = s if s is not None else float(v[i])
                if   score < best - 1e-6:        best = score; pool = [mv]
                elif abs(score - best) < 1e-6:   pool.append(mv)
            return self.rng.choice(pool)

    # ====================================================================
    # 3. LLM agent (OpenRouter Claude Opus 4.6, with full repair pipeline)
    # ====================================================================
    SYSTEM_PROMPT = (
        "You are a chess engine. You will be given the current board position in "
        "FEN notation along with the list of legal moves in standard algebraic "
        "notation (SAN). Reply with EXACTLY ONE legal move in SAN. Do not include "
        "any other text, commentary, punctuation, or move number. Choose the move "
        "that is best for the side to move."
    )
    _PREAMBLE_RE = re.compile(
        r'^(my move is|i play|i would play|i choose|the best move is|move:?)\\s*',
        flags=re.IGNORECASE,
    )

    def _build_user_prompt(board) -> str:
        legal_san = sorted(board.san(m) for m in board.legal_moves)
        side = 'White' if board.turn == chess.WHITE else 'Black'
        return (f'Position (FEN): {board.fen()}\\n'
                f'Side to move: {side}\\n'
                f'Legal moves ({len(legal_san)}): ' + ' '.join(legal_san) +
                '\\nReply with one SAN move only.')

    def _parse_san(text, board):
        if not text: return None
        s = text.strip().strip('."\\'`')
        s = _PREAMBLE_RE.sub('', s).strip()
        s = (s.split()[0] if s.split() else s).strip('.,;:!?')
        try: return board.parse_san(s)
        except Exception:
            try:
                mv = chess.Move.from_uci(s.lower())
                return mv if mv in board.legal_moves else None
            except ValueError: return None

    class LLMAgent:
        def __init__(self, model: Optional[str] = None, seed: int = 0):
            self.openrouter_key = os.environ.get('OPENROUTER_API_KEY', '')
            self.anthropic_key  = os.environ.get('ANTHROPIC_API_KEY',  '')
            self.model = model or os.environ.get('COMP532_LLM_MODEL', 'anthropic/claude-opus-4.6')
            self.max_retries = 2
            self._fallback = HeuristicAgent(seed=seed)
            if not (self.openrouter_key or self.anthropic_key):
                raise RuntimeError(
                    'LLMAgent requires OPENROUTER_API_KEY or ANTHROPIC_API_KEY.')

        def _call(self, board, hint=''):
            prompt = _build_user_prompt(board) + (f'\\n\\nNOTE: {hint}' if hint else '')
            if self.openrouter_key:
                from openai import OpenAI
                client = OpenAI(api_key=self.openrouter_key,
                                base_url='https://openrouter.ai/api/v1')
                resp = client.chat.completions.create(
                    model=self.model,
                    messages=[{'role':'system','content':SYSTEM_PROMPT},
                              {'role':'user','content':prompt}],
                    temperature=0.2, max_tokens=8,
                    extra_headers={
                        'HTTP-Referer': 'https://github.com/comp532-cw2-chess',
                        'X-Title': 'COMP532 chess bot',
                    },
                )
                return resp.choices[0].message.content or ''
            else:
                import anthropic
                c = anthropic.Anthropic(api_key=self.anthropic_key)
                r = c.messages.create(
                    model=self.model, system=SYSTEM_PROMPT,
                    messages=[{'role':'user','content':prompt}],
                    max_tokens=12, temperature=0.2,
                )
                return ''.join(b.text for b in r.content if getattr(b, 'type', None) == 'text')

        def select_move(self, board):
            hint = ''
            for _ in range(self.max_retries + 1):
                try:
                    text = self._call(board, hint=hint)
                except Exception as e:
                    print(f'[Comp532Bot] LLM call failed: {e}; using heuristic fallback')
                    return self._fallback.select_move(board)
                mv = _parse_san(text, board)
                if mv is not None: return mv
                hint = f"Your previous response {text!r} was not a legal SAN move."
            print('[Comp532Bot] LLM exhausted retries; using heuristic fallback')
            return self._fallback.select_move(board)

    # ====================================================================
    # 4. The lichess-bot homemade engine class
    # ====================================================================
    AGENT_NAME = os.environ.get('COMP532_AGENT', 'heuristic').lower()
    DRL_MODEL_PATH = os.environ.get(
        'COMP532_DRL_MODEL', 'models/chess_drl.pt')

    def _make_agent():
        if AGENT_NAME == 'heuristic': return HeuristicAgent(seed=0)
        if AGENT_NAME == 'drl':       return DRLAgent(model_path=DRL_MODEL_PATH)
        if AGENT_NAME == 'llm':       return LLMAgent()
        raise ValueError(f'Unknown COMP532_AGENT={AGENT_NAME!r}; '
                         'use heuristic|drl|llm')

    class Comp532Bot(MinimalEngine):
        """Homemade lichess-bot engine that wraps a COMP532 chess agent.

        lichess-bot's contract: implement search(board, time_limit, ponder,
        draw_offered, root_moves) and return either a chess.Move (older API)
        or a chess.engine.PlayResult (current API). We return PlayResult when
        importable and a bare Move otherwise.
        """
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs) if MinimalEngine is not object else None
            self.agent = _make_agent()
            print(f'[Comp532Bot] ready: agent={AGENT_NAME}')

        def search(self, board, time_limit, ponder, draw_offered, root_moves):
            mv = self.agent.select_move(board)
            if PlayResult is not None:
                return PlayResult(mv, None)
            return mv
''')

with open('homemade.py', 'w') as f:
    f.write(HOMEMADE_PY)

print(f"Wrote homemade.py ({len(HOMEMADE_PY):,} chars)")
print("Copy this file into your lichess-bot/ directory.")

### Cell 2 — write `config.yml` (run once)

This produces a minimal `config.yml`. Copy it into your `lichess-bot/` clone, replacing the default one. The token is read from the `LICHESS_BOT_TOKEN` environment variable — **the file itself contains no secret**, which makes it safe to commit if you wish.

In [ ]:
from textwrap import dedent

CONFIG_YML = dedent('''\
    # config.yml for COMP532 chess bot.
    # Reads the lichess BOT token from the LICHESS_BOT_TOKEN env var; never
    # paste a token directly into this file.

    token: ""            # leave blank; lichess-bot reads $LICHESS_BOT_TOKEN

    url: "https://lichess.org/"

    engine:
      dir: "."
      name: "Comp532Bot"             # class name in homemade.py
      protocol: "homemade"            # use the Python homemade-engine path
      ponder: false
      polyglot:
        enabled: false
      draw_or_resign:
        resign_enabled: false
        offer_draw_enabled: false
      online_moves:
        max_out_of_book_moves: 0
        max_retries: 0
      uci_options: {}
      homemade_options: {}

    abort_time: 30
    fake_think_time: false
    rate_limiting_delay: 0

    correspondence:
      checkin_period: 600
      move_time: 60
      disconnect_time: 300

    challenge:
      concurrency: 1
      sort_by: "best"
      accept_bot: true
      only_bot: false
      max_increment: 60
      min_increment: 0
      max_base: 1800
      min_base: 30
      variants:
        - standard
      time_controls:
        - bullet
        - blitz
        - rapid
        - classical
      modes:
        - casual
        - rated
''')

with open('config.yml', 'w') as f:
    f.write(CONFIG_YML)

print(f"Wrote config.yml ({len(CONFIG_YML):,} chars)")
print("Copy this file into your lichess-bot/ directory, replacing config.yml.")

### Cell 3 — start the bot (run from a terminal, **not** from the notebook)

From inside your `lichess-bot/` clone:

```bash
# 1. Set the credentials in your shell (do this once per terminal session)
export LICHESS_BOT_TOKEN='lip_…'                 # your bot's lichess OAuth token
export OPENROUTER_API_KEY='sk-or-v1-…'           # only if you'll use the LLM agent
export COMP532_AGENT='heuristic'                 # or 'drl' or 'llm'
export COMP532_DRL_MODEL='/path/to/chess_drl.pt' # only for COMP532_AGENT=drl

# 2. Verify the bot is registered with lichess:
curl -H "Authorization: Bearer $LICHESS_BOT_TOKEN" \\
     https://lichess.org/api/account/me | python -m json.tool
# Expect to see  "title": "BOT"  in the response.

# 3. Install lichess-bot's own dependencies (one-off)
python3 -m venv venv && source venv/bin/activate
pip install -r requirements.txt
pip install openai numpy torch python-chess        # for our agents

# 4. Run
python lichess-bot.py
```

Now any human (or other bot) can challenge your bot from its lichess profile page at `https://lichess.org/@/<your-bot-username>`. The bot will accept according to the `challenge:` block in `config.yml` and play with whichever agent you set in `COMP532_AGENT`.

**Why not chess.com?** Chess.com offers no public bot/play API. The two ways a program could play there in practice are (i) authenticated *browser automation* against the chess.com web UI, and (ii) reverse-engineering the private REST endpoints used by their own front-end. Both violate the chess.com Terms of Service. Lichess is the supported equivalent.